# Model Informations

In [ ]:
# params
k = 2, l = 2

# iteration 1

dataset = [
  ['A', 'B', 'C', 'y'],
  [ 0 ,  1 ,  0 ,  0 ],
  [ 1 ,  0 ,  1 ,  1 ],
  [ 0 ,  1 ,  0 ,  1 ],
  [ 0,   0,   0,   1 ]
]

rules = ('A' and '¬B')

''' propositional variables to create the rule
bA,1,1 = 1 bA,1,2 = 0
bB,1,1 = 0 bB,1,2 = 1
bC,1,1 = 0 bC,1,2 = 0

u*1,1 = 0  u*1,2 = 0
p1,1 = 1   p1,2 = 0
'''

# iteration 2

dataset = [
  ['A', 'B', 'C', 'y'],
  [ 0 ,  1 ,  0 ,  1 ],
  [ 0,   0,   0,   1 ]
]

# OBS:
''' restrictions to create an inconsistent rule starting from the second iteration

- for each rule that already exists in the set of rules do:
(bA,1,1 and ¬p1,1) or 
(bA,1,2 and ¬p1,2) or 
(bB,1,1 and p1,1) or 
(bB,1,2 and p1,2)

- converting to CNF using Tseytin
(x or y or z or w) and

//** x -> (bA,1,1 and ¬p1,1) **//
(¬x or bA,1,1) and
(¬x or ¬p1,1) and

(¬y or bA,1,2) and
(¬y or ¬p1,2) and

(¬z or bB,1,1) and
(¬z or p1,1) and

(¬w or bB,1,2) and
(¬w or p1,2)

'''

# Model Tests

In [1]:
import sys, os
if not sys.path[0] == os.path.abspath('..'):
    sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
from models.imlib import IMLIB
from models.i_imlib import I_IMLIB
from models.di_imlib import DI_IMLIB
from models.di_imlib_m import DI_IMLIB_M
from sklearn.model_selection import train_test_split


Xy = pd.read_csv('../databases/lung_cancer.csv')

X = Xy.drop(['Class'], axis=1)
y = Xy['Class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2)

In [2]:
di_model = DI_IMLIB(
    max_rule_set_size=3,
    max_size_each_rule=4,
    categorical_columns_index=[0, 1],
    number_quantiles_ordinal_columns=2,
    number_lines_per_partition=8,
    rules_accuracy_weight=10,
    balance_instances=True,
    # balance_instances_seed=21
)
# di_m_model = DI_IMLIB_M(
#     max_rule_set_size=3,
#     max_size_each_rule=4,
#     categorical_columns_index=[],
#     number_quantiles_ordinal_columns=4,
#     number_lines_per_partition=8,
#     rules_accuracy_weight=10,
#     balance_instances=True,
#     # balance_instances_seed=21
# )

di_model.fit(X, y)
# di_m_model.fit(X,y)

In [3]:
line_instance = 5
instance = Xy.iloc[line_instance - 2].values[:-1]
print(f'Rules: {di_model.get_rules()}')
print(f'Instance: {[feat + ": " + str(instance[i_feat]) for i_feat, feat in enumerate(Xy.columns.values[:-1])]}')
print(f'Predict: {di_model.predict(instance)}')
print(f'Sufficient reason: {di_model.get_sufficient_reasons(instance)}')

Rules: (Not Surname Jackson and Not Surname Anderson and AreaQ <= 5.0)
Instance: ['Name: Alex', 'Surname: Telles', 'Age: 28', 'Smokes: 0', 'AreaQ: 8', 'Alkhol: 1']
Predict: 0
Sufficient reason: (AreaQ <= 5.0)


In [4]:
di_model.get_dataset_binarized().get_original_to_binarized_values()[0]['John']

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0])

In [5]:
di_model.get_dataset_binarized().get_qtts_binarized_feat_per_original_feat()

<IntegerArray>
[49, 49, 1, 1, 1, 1]
Length: 6, dtype: Int64

In [6]:
di_model.get_dataset_binarized().get_range_binarized_columns()

<NumpyExtensionArray>
[(1, 49), (50, 98), (99, 99), (100, 100), (101, 101), (102, 102)]
Length: 6, dtype: object

In [7]:
def get_bin_cols_by_orig_cols_range(index_original_columns, list_dict_vars_vals, columns_range):
    list_bin_cols_same_range = [[] for _ in index_original_columns]
    for i_cat_col in index_original_columns:
        for bin_col in list_dict_vars_vals[0]:
            if bin_col >= columns_range[i_cat_col][0] and bin_col <= columns_range[i_cat_col][1]:
                list_bin_cols_same_range[i_cat_col].append(bin_col)
    
    for list in list_bin_cols_same_range:
        list.sort()

    return list_bin_cols_same_range

def remove_invalid_combinations_categorical_variables(
        categorical_index_original_columns, 
        list_dict_vars_vals, 
        columns_range,
        dict_original_to_binarized_values
    ):
    list_bin_cols_same_range = get_bin_cols_by_orig_cols_range(categorical_index_original_columns, list_dict_vars_vals, columns_range)
    new_list_dict_vars_vals = []

    for bin_cols in list_bin_cols_same_range:
        if len(bin_cols) > 1:
            i_bin_cols = [n - 1 for n in bin_cols]

            for dict_vars_vals in list_dict_vars_vals:
                comb_vals = [dict_vars_vals[bin] for bin in bin_cols]

                for key in dict_original_to_binarized_values:
                    # in this moment, we verify if the combination exists
                    if np.all(
                        dict_original_to_binarized_values[key][i_bin_cols] == comb_vals
                    ):
                        new_list_dict_vars_vals.append(dict_vars_vals)

    return new_list_dict_vars_vals

categorical_index_original_columns = [0, 1]
list_dict_vars_vals = [{51:0, 50:0}, {51:0, 50:1}, {51:1, 50:0}, {51:1, 50:1}]
columns_range = [(1, 49), (50, 98), (99, 99), (100, 100), (101, 101), (102, 102)]
dict_original_to_binarized_values = di_model.get_dataset_binarized().get_original_to_binarized_values()[0]

# get_bin_cols_by_orig_cols_range(categorical_index_original_columns, list_dict_vars_vals, columns_range)
remove_invalid_combinations_categorical_variables(
    categorical_index_original_columns, 
    list_dict_vars_vals, 
    columns_range,
    dict_original_to_binarized_values
)


IndexError: index 49 is out of bounds for axis 0 with size 49

In [18]:
dict = {1:0, 2:3}
del dict[1]
dict

{2: 3}

In [6]:
di_model.get_dataset_binarized().get_normal_features_label()

<NumpyExtensionArray>
[    'Name Alec ',      'Name Alex',      'Name Anna',    'Name Barbra',
   'Name Barbra ',    'Name Camela', 'Name Charlize ', 'Name Charlton ',
 'Name Cristiano',    'Name Diane ',
 ...
    'Surname Tal', 'Surname Telles', 'Surname Theron',  'Surname Wayne',
   'Surname Wick',  'Surname Wyman',    'Age <= 39.0', 'Smokes <= 15.0',
   'AreaQ <= 5.0',  'Alkhol <= 3.0']
Length: 102, dtype: object

In [ ]:
print('DI-IMLIB:')
print(di_model.get_rules())
print()
# print('DI-IMLIB-M:')
# print(di_m_model.get_rules())

DI-IMLIB:
(Not Name Alex and Alkhol > 3.0)



In [8]:
print('DI-IMLIB:', di_model.score(X, y))
# print('DI-IMLIB-M:', di_m_model.score(X, y))

DI-IMLIB: 0.4915254237288136


In [5]:
print(di_model.get_rules_size())
print(di_m_model.get_rules_size())

[4]
{np.int64(0): [1, 2], np.int64(1): [4]}


In [17]:
print(di_model.get_rule_set_size())
print(di_m_model.get_rule_set_size())

2
{np.int64(0): 2, np.int64(1): 1}


In [18]:
print(di_model.get_larger_rule_size())
print(di_m_model.get_larger_rule_size())

3
{np.int64(0): 2, np.int64(1): 3}


In [19]:
print(di_model.get_sum_rules_size())
print(di_m_model.get_sum_rules_size())

5
{np.int64(0): 3, np.int64(1): 3}


In [33]:
import re

def remove_redundancias(literals):
    parsed = []
    for literal in literals:
        match = re.match(r"(\w+)\s*([<>]=?)\s*(-?\d+)", literal)
        if match:
            var, op, value = match.groups()
            value = int(value)
            parsed.append((var, op, value, literal))

    reduced = {}
    
    for var, op, value, literal in parsed:
        if var not in reduced:
            reduced[var] = []
        reduced[var].append((op, value, literal))

    final_literals = set(literals)
    
    for var, conditions in reduced.items():
        conditions.sort(key=lambda x: x[1])  # Ordena pelo valor numérico
        
        to_remove = set()
        for i in range(len(conditions) - 1):
            op1, val1, lit1 = conditions[i]
            op2, val2, lit2 = conditions[i + 1]

            if op1 == "<=" and op2 == "<=":
                to_remove.add(lit2)
            elif op1 == ">=" and op2 == ">=":
                to_remove.add(lit1)
            elif op1 == ">" and op2 == ">":
                to_remove.add(lit1)
            elif op1 == "<" and op2 == "<":
                to_remove.add(lit2)

        final_literals -= to_remove
    
    return list(final_literals)

# Exemplo de uso:
literals = ['idade <= 20', 'idade <= 16.9', 'idade <= 17', 'salario > 300', 'salario > 500']
print(remove_redundancias(literals))

['idade <= 16.9', 'salario > 500']
